<a href="https://colab.research.google.com/github/parker-group/earth-observation-howto/blob/main/ERA5_Hourly_Stations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# ERA5-Land hourly point extraction
#
# Extracts:
#   1. 2 m air temperature, degrees Celsius
#   2. 2 m dew-point temperature, degrees Celsius
#
# Time period:
# June 30, 2026 through the latest ERA5-Land hour currently available in GEE
#
# Output:
#   One CSV row per station per hour.
# ============================================================================


import ee
import pandas as pd
from pathlib import Path


# ----------------------------------------------------------------------------
# 1. authenticate and start GEE
# ----------------------------------------------------------------------------

# Replace this with the Google Cloud project used for Earth Engine.
GEE_PROJECT = "YOUR-GOOGLE-CLOUD-PROJECT"

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)


# ----------------------------------------------------------------------------
# 2. provide station coords
# ----------------------------------------------------------------------------

# Set this to False to use the coordinates entered manually below.
# Set this to True to read coordinates from a local CSV.

USE_CSV = False


# Option A: manually entered coordinates

manual_stations = pd.DataFrame(
    [
        {
            "station": "B1",
            "latitude": 33.53967,
            "longitude": -115.94629,
        },
        {
            "station": "B2",
            "latitude": 33.59750,
            "longitude": -116.22480,
        },
        {
            "station": "L1",
            "latitude": 33.61097,
            "longitude": -116.23455,
        },
        {
            "station": "L2",
            "latitude": 33.59817,
            "longitude": -116.22505,
        },
        {
            "station": "L3",
            "latitude": 33.54228,
            "longitude": -115.94064,
        },
        {
            "station": "L4",
            "latitude": 33.53968,
            "longitude": -115.94721,
        },
    ]
)


# Option B: coordinates supplied in a CSV
#
# The CSV must contain these columns:
#
# station,latitude,longitude
# B1,33.53967,-115.94629
# B2,33.59750,-116.22480

INPUT_CSV = "/content/stations.csv"


if USE_CSV:
    stations = pd.read_csv(INPUT_CSV)
else:
    stations = manual_stations.copy()


required_columns = {"station", "latitude", "longitude"}

missing_columns = required_columns.difference(stations.columns)

if missing_columns:
    raise ValueError(
        "The station table is missing these required columns: "
        + ", ".join(sorted(missing_columns))
    )


stations = stations[
    ["station", "latitude", "longitude"]
].copy()


if stations.empty:
    raise ValueError("The station table contains no locations.")


if stations["station"].duplicated().any():
    duplicate_names = stations.loc[
        stations["station"].duplicated(),
        "station",
    ].tolist()

    raise ValueError(
        "Station names must be unique. Duplicate names: "
        + ", ".join(map(str, duplicate_names))
    )


print("Stations to extract:")
print(stations.to_string(index=False))


# ----------------------------------------------------------------------------
# 3. set date range
# ----------------------------------------------------------------------------

PACIFIC_TIME_ZONE = "America/Los_Angeles"

# Midnight Pacific time at the beginning of June 30, 2026.

start_date = ee.Date(
    "2026-06-30T00:00:00",
    PACIFIC_TIME_ZONE,
)


# Load the complete ERA5-Land hourly collection.

era5_all = (
    ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
    .select(
        [
            "temperature_2m",
            "dewpoint_temperature_2m",
        ]
    )
)


# Find the latest ERA5-Land hour currently available.

latest_time_ms = ee.Number(
    era5_all.aggregate_max("system:time_start")
)

latest_date = ee.Date(latest_time_ms)


# filterDate excludes the ending timestamp.
# Adding one hour includes the latest available image.

end_date = latest_date.advance(1, "hour")


print(
    "\nRequested start:",
    start_date.format(
        "YYYY-MM-dd HH:mm",
        PACIFIC_TIME_ZONE,
    ).getInfo(),
    "Pacific time",
)

print(
    "Latest available:",
    latest_date.format(
        "YYYY-MM-dd HH:mm",
        PACIFIC_TIME_ZONE,
    ).getInfo(),
    "Pacific time",
)

print(
    "Latest available:",
    latest_date.format(
        "YYYY-MM-dd HH:mm",
        "UTC",
    ).getInfo(),
    "UTC",
)

# ----------------------------------------------------------------------------
# 4. convert temp and dew point to celsius
# ----------------------------------------------------------------------------

def prepare_image(image):
    """
    Convert ERA5-Land air temperature and dew point from Kelvin to Celsius.
    """

    image = ee.Image(image)

    temperature_c = (
        image.select("temperature_2m")
        .subtract(273.15)
        .rename("temperature_2m_c")
    )

    dewpoint_c = (
        image.select("dewpoint_temperature_2m")
        .subtract(273.15)
        .rename("dewpoint_temperature_2m_c")
    )

    return (
        ee.Image.cat(
            [
                temperature_c,
                dewpoint_c,
            ]
        )
        .copyProperties(
            image,
            ["system:time_start"],
        )
    )


hourly_data = (
    era5_all
    .filterDate(start_date, end_date)
    .map(prepare_image)
    .sort("system:time_start")
)


number_of_hours = hourly_data.size().getInfo()
expected_rows = number_of_hours * len(stations)

print(f"\nNumber of ERA5-Land hours: {number_of_hours:,}")
print(f"Number of stations: {len(stations):,}")
print(f"Expected output rows: {expected_rows:,}")


if number_of_hours == 0:
    raise RuntimeError(
        "No ERA5-Land images were found for the requested date range."
    )



# ----------------------------------------------------------------------------
# 5. extract the hourly time series for each stations, one at a time
# ----------------------------------------------------------------------------

# The requests are separated by station. Each request returns only
# one station's hourly series rather than constructing one large nested table.

station_results = []


for station_row in stations.itertuples(index=False):

    station_name = str(station_row.station)
    latitude = float(station_row.latitude)
    longitude = float(station_row.longitude)

    point = ee.Geometry.Point(
        [longitude, latitude]
    )


    def extract_hour(image):
        """
        Extract the two band values from one hourly image at this point.
        """

        image = ee.Image(image)

        image_date = ee.Date(
            image.get("system:time_start")
        )

        values = image.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=10_000,
            maxPixels=100_000,
        )

        return ee.Feature(
            None,
            values,
        ).set(
            {
                "station": station_name,
                "latitude": latitude,
                "longitude": longitude,

                "time_start_utc_ms": image_date.millis(),

                "datetime_utc": image_date.format(
                    "YYYY-MM-dd'T'HH:mm:ss",
                    "UTC",
                ),

                "datetime_pacific": image_date.format(
                    "YYYY-MM-dd'T'HH:mm:ss",
                    PACIFIC_TIME_ZONE,
                ),
            }
        )


    station_features = hourly_data.map(
        extract_hour
    )


    # Download this station's FeatureCollection directly as a pandas DataFrame.
    # Earth Engine automatically retrieves all pages of the result.

    station_dataframe = ee.data.computeFeatures(
        {
            "expression": station_features,
            "fileFormat": "PANDAS_DATAFRAME",
            "pageSize": 1000,
        }
    )


    station_results.append(
        station_dataframe
    )

    print(
        f"Retrieved {len(station_dataframe):,} hours for {station_name}"
    )


# ----------------------------------------------------------------------------
# 6. combine and clean results
# ----------------------------------------------------------------------------

hourly_station_data = pd.concat(
    station_results,
    ignore_index=True,
)


output_columns = [
    "station",
    "latitude",
    "longitude",
    "time_start_utc_ms",
    "datetime_utc",
    "datetime_pacific",
    "temperature_2m_c",
    "dewpoint_temperature_2m_c",
]


missing_output_columns = [
    column
    for column in output_columns
    if column not in hourly_station_data.columns
]

if missing_output_columns:
    raise RuntimeError(
        "The Earth Engine result is missing these columns: "
        + ", ".join(missing_output_columns)
    )


hourly_station_data = hourly_station_data[
    output_columns
].copy()


hourly_station_data["temperature_2m_c"] = pd.to_numeric(
    hourly_station_data["temperature_2m_c"],
    errors="coerce",
)

hourly_station_data["dewpoint_temperature_2m_c"] = pd.to_numeric(
    hourly_station_data["dewpoint_temperature_2m_c"],
    errors="coerce",
)


hourly_station_data = hourly_station_data.sort_values(
    [
        "station",
        "time_start_utc_ms",
    ]
).reset_index(drop=True)


print(
    f"\nFinal output contains {len(hourly_station_data):,} rows."
)

print("\nFirst 12 rows:")
print(
    hourly_station_data.head(12).to_string(index=False)
)

# ----------------------------------------------------------------------------
# 7. SAVE THE RESULTS AS A CSV
# ----------------------------------------------------------------------------

output_csv = Path(
    "/content/ERA5L_CoachV_hourly.csv"
)

hourly_station_data.to_csv(
    output_csv,
    index=False,
)

print(f"\nSaved CSV to: {output_csv}")


# Optional in Google Colab:
#
# from google.colab import files
# files.download(str(output_csv))

Stations to extract:
station  latitude  longitude
     B1  33.53967 -115.94629
     B2  33.59750 -116.22480
     L1  33.61097 -116.23455
     L2  33.59817 -116.22505
     L3  33.54228 -115.94064
     L4  33.53968 -115.94721

Requested start: 2026-06-30 00:00 Pacific time
Latest available: 2026-07-17 15:00 Pacific time
Latest available: 2026-07-17 22:00 UTC

Number of ERA5-Land hours: 386
Number of stations: 6
Expected output rows: 2,316
Retrieved 386 hours for B1
Retrieved 386 hours for B2
Retrieved 386 hours for L1
Retrieved 386 hours for L2
Retrieved 386 hours for L3
Retrieved 386 hours for L4

Final output contains 2,316 rows.

First 12 rows:
station  latitude  longitude  time_start_utc_ms        datetime_utc    datetime_pacific  temperature_2m_c  dewpoint_temperature_2m_c
     B1  33.53967 -115.94629      1782802800000 2026-06-30T07:00:00 2026-06-30T00:00:00         25.205698                  11.306238
     B1  33.53967 -115.94629      1782806400000 2026-06-30T08:00:00 2026-06-30T0